# Export MRLFADS Submission Artifact

This notebook loads an MRLFADS run from checkpoint, executes validation to populate model outputs, and exports a benchmark submission artifact H5.

Exported artifact fields:
- `area-A*`: predicted activity means (same shape as truth activity)
- `message-mesgs`: concatenated communication slices (optional metric input)
- `message-latents`: concatenated factor states (optional diagnostic)


In [15]:
from pathlib import Path
import sys
import json
import shutil

import h5py
import yaml
import numpy as np
import pandas as pd


for base in [Path.cwd(), *Path.cwd().parents]:
    if (base / "src").exists() and (base / "mrlfads2").exists():
        repo_root = base
        break
else:
    raise FileNotFoundError("Could not locate repo root containing src/ and mrlfads2/")

sys.path.insert(0, str(repo_root / "mrlfads2"))
sys.path.insert(0, str(repo_root / "src"))

repo_root

PosixPath('/Users/william_ong/Desktop/Projects/dgn')

In [16]:
# ----- User inputs -----
run_dir = repo_root / "runs" / "mn_fit_pois_01"
config_path = run_dir / "configs" / "main.yaml"

# Use latest checkpoint by default. You can set a specific file if desired.
checkpoint_name = None  # e.g., "339-18700.ckpt"

# Ground-truth file used to align datamodule path expectation.
truth_h5 = repo_root / "datasets" / "memory_network" / "poisson.h5"

# Output submission artifact path.
submission_h5 = run_dir / "submission_artifact.h5"

assert run_dir.exists(), f"Missing run_dir: {run_dir}"
assert config_path.exists(), f"Missing config_path: {config_path}"
assert truth_h5.exists(), f"Missing truth_h5: {truth_h5}"

print("run_dir:", run_dir)
print("config_path:", config_path)
print("truth_h5:", truth_h5)
print("submission_h5:", submission_h5)

run_dir: /Users/william_ong/Desktop/Projects/dgn/runs/mn_fit_pois_01
config_path: /Users/william_ong/Desktop/Projects/dgn/runs/mn_fit_pois_01/configs/main.yaml
truth_h5: /Users/william_ong/Desktop/Projects/dgn/datasets/memory_network/poisson.h5
submission_h5: /Users/william_ong/Desktop/Projects/dgn/runs/mn_fit_pois_01/submission_artifact.h5


In [17]:
# MRLFADS BasicDataModule expects: <mrlfads.paths.datapath>/<filename>/data.h5
# This run's datamodule config filename is typically memory_network_data_mem2_rank1.
# We create an alias dataset folder if needed.

dm_cfg_path = run_dir / "configs" / "datamodule" / "datamodule.yaml"
with dm_cfg_path.open("r", encoding="utf-8") as f:
    dm_cfg = yaml.safe_load(f)
expected_dataset_name = dm_cfg["filename"]

alias_dir = repo_root / "datasets" / expected_dataset_name
alias_dir.mkdir(parents=True, exist_ok=True)
alias_h5 = alias_dir / "data.h5"

if not alias_h5.exists():
    try:
        alias_h5.symlink_to(truth_h5)
        print(f"Created symlink: {alias_h5} -> {truth_h5}")
    except OSError:
        shutil.copy2(truth_h5, alias_h5)
        print(f"Symlink failed; copied file to: {alias_h5}")
else:
    print(f"Alias already exists: {alias_h5}")

alias_h5

Alias already exists: /Users/william_ong/Desktop/Projects/dgn/datasets/memory_network_poisson_250hz/data.h5


PosixPath('/Users/william_ong/Desktop/Projects/dgn/datasets/memory_network_poisson_250hz/data.h5')

In [18]:

import mrlfads.paths as mpaths
import mrlfads.datamodules as mdm
from mrlfads.run import load

# Force MRLFADS data path to this local repo dataset root.
mpaths.datapath = str(repo_root / "datasets")
mdm.path.datapath = mpaths.datapath

state = load(
    config_path=str(config_path),
    validate=True,
    use_best=False,
)

model = state["model"]
model.eval()
print("Loaded model areas:", model.area_names)
print("Validation metrics keys (sample):", list(state["metrics"].keys())[:8])

Global seed set to 42


Config path:  /Users/william_ong/Desktop/Projects/dgn/runs/mn_fit_pois_01/configs/main.yaml
Checkpoint path:  /Users/william_ong/Desktop/Projects/dgn/runs/mn_fit_pois_01


GPU available: True (mps), used: False
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs
/Users/william_ong/Desktop/Projects/dgn/.venv/lib/python3.10/site-packages/pytorch_lightning/trainer/setup.py:201: UserWarning: MPS available but not used. Set `accelerator` and `devices` using `Trainer(accelerator='mps', devices=1)`.
  rank_zero_warn(


Validation DataLoader 0: 100%|██████████| 1/1 [00:02<00:00,  2.90s/it]
Loaded model areas: ['A0', 'A1', 'A2']
Validation metrics keys (sample): ['valid/A0/recon', 'valid/A0/l2', 'valid/A0/kl/ic', 'valid/A0/kl/co', 'valid/A0/kl/com', 'valid/A0/kl/gv', 'valid/A0/r2', 'valid/A0/hn']


In [19]:
# Build export arrays from session 0.
session = 0
area_arrays = {}
comm_slices = []
latent_slices = []

for area_name in model.area_names:
    area = model.areas[area_name]
    ahps = area.hparams

    # Model readout output stores distribution parameters.
    # Convert to predictive mean via output_dist.unbind + output_dist.compute_means.
    raw_pred = model.outputs[area_name][session].detach().cpu()
    pred_params = area.output_dist.unbind(raw_pred)
    pred_mean = area.output_dist.compute_means(pred_params).detach().cpu().numpy()
    area_arrays[f"area-{area_name}"] = pred_mean

    # Communication slices from decoder inputs: [ci_enc | communication | co]
    inp = model.save_var[area_name].inputs.detach().cpu().numpy()
    ci = ahps.ci_enc_dim
    cm = ahps.com_dim * model.hparams.num_other_areas
    comm_slices.append(inp[:, :, ci:ci + cm])

    # Factor states as optional latent diagnostic.
    states = model.save_var[area_name].states.detach().cpu().numpy()
    latent_slices.append(states[:, 1:, -ahps.fac_dim:])

message_mesgs = np.concatenate(comm_slices, axis=-1)
message_latents = np.concatenate(latent_slices, axis=-1)

print({k: v.shape for k, v in area_arrays.items()})
print("message-mesgs:", message_mesgs.shape)
print("message-latents:", message_latents.shape)

{'area-A0': (153, 190, 64), 'area-A1': (153, 190, 64), 'area-A2': (153, 190, 64)}
message-mesgs: (153, 190, 72)
message-latents: (153, 190, 192)


In [20]:
if submission_h5.exists():
    submission_h5.unlink()

# Export metadata to let evaluator align with truth file shape.
val_indices = np.asarray(state["datamodule"].val_session_indices[session], dtype=int)
time_start = int(model.hparams.ic_enc_seq_len)

with h5py.File(submission_h5, "w") as h5:
    g = h5.create_group("0")

    for key, arr in area_arrays.items():
        ds = g.create_dataset(key, data=arr)
        ds.attrs["type"] = "prediction"

    ds = g.create_dataset("message-mesgs", data=message_mesgs)
    ds.attrs["type"] = "prediction"

    ds = g.create_dataset("message-latents", data=message_latents)
    ds.attrs["type"] = "prediction"

    # Alignment metadata for evaluator
    g.create_dataset("meta-batch-indices", data=val_indices)
    g.create_dataset("meta-time-start", data=np.array([time_start], dtype=np.int32))

    g.attrs["source_run_dir"] = str(run_dir)
    g.attrs["source_config"] = str(config_path)
    g.attrs["session"] = str(session)
    g.attrs["model_type"] = "mrlfads"

print("Wrote submission artifact:", submission_h5)
print("meta-batch-indices shape:", val_indices.shape)
print("meta-time-start:", time_start)

Wrote submission artifact: /Users/william_ong/Desktop/Projects/dgn/runs/mn_fit_pois_01/submission_artifact.h5
meta-batch-indices shape: (153,)
meta-time-start: 10


In [22]:
# Optional sanity-check with benchmark evaluator.
from evals import evaluate_memory_network_h5_submission

res = evaluate_memory_network_h5_submission(
    pred_h5_path=submission_h5,
    truth_h5_path=truth_h5,
    config_dir=repo_root / "configs" / "memory_network",
    session="0",
)

print("Metric status:", json.dumps(res.metric_status, indent=2))
pd_summary = pd.DataFrame([res.summary]).T.rename(columns={0: "value"})
pd_summary.head(12)

# Full decodability table
res.communication_decode.head(20)

# Optional: matrix view (target area x source component)
res.communication_decode.pivot(
    index="target_area",
    columns="source_component",
    values="decode_r2",
)

Metric status: {
  "reconstruction": "ok",
  "communication_decode": "ok",
  "message_recovery": "ok"
}


source_component,0,1,2
target_area,,,
A0,0.964172,-0.003600,0.887990
A1,0.745452,0.959865,-0.005075
A2,-0.003541,0.712233,0.961640
